In [1]:
import sys,os
import traceback

from Database.DB_reader import Database
from Strategies.Autotrader.TP_api import TP_api
import Strategies.Autotrader.enumerate as ENUM
import pandas as pd
from datetime import datetime, timezone


In [2]:
from Utilities.email_sending import send_plain_email, send_html_email
EMAIL_PASSWORD = os.getenv('EMAIL_PASSWORD') # the password needs to be set as EMAIL_PASSWORD in system variables of the computer where the process is running, it is located in S:/Algo/email_password.txt
if EMAIL_PASSWORD is None:
    raise ValueError("EMAIL_PASSWORD environment variable not set")

RECIPIENT = ["scasny_martin@energytrading.sk", "zubal_andrej@energytrading.sk"] # can also be a list of recipients

ValueError: EMAIL_PASSWORD environment variable not set

In [3]:
db_cols = {'trade_id',
           'order_id',
           'exchange',
           'execution_time',
           'state',
           'trading_portfolio',
           'price',
           'quantity',
           'buy_delivery_area',
           'sell_delivery_area',
           'product_id',
           'product_type',
           'product_name',
           'slot_type',
           'slot_information',
           'counterparty',
           'aggressor',
           'initiator',
           'aggressor_broker_id',
           'initiator_broker_id'}

In [4]:
if __name__ == "__main__":
    db_r = Database()
    cls = TP_api('prod')
    today = datetime.now()
    today = datetime(2024,5,23)

    try:
        ts_utc = datetime(today.year, today.month, today.day, 0, 0, 0).isoformat(timespec='milliseconds')
        print(ts_utc)
        result_dict = cls.get_own_trades(ts_utc)

        result_dictlist = [d for d in result_dict if d['trader_name'] == '220_ETC-autotrader']
        filtered_data = [{col: data[col] for col in db_cols if col in data} for data in result_dictlist]
        for data_dict in filtered_data:
            data_dict['algo_id'] = data_dict.pop('trading_portfolio', None)
        
        df = pd.DataFrame(filtered_data)
        r = df.to_sql(schema='algo', name='stage_strategy_trades', if_exists='replace', con=db_r.connection_string, index=False)
        print(r)
        db_r.merge_from_staging_to_prod_enum('algo', 'strategy_trades')
    except Exception as e:
        error_message = str(e)

        # Capture the traceback
        error_traceback = traceback.format_exc()
        html_content = f"""\
        <html>
            <body>
                <h1>Failed job strategy_trades_daily_update </h1>
                <p>{error_message}</p>
                <p>{error_traceback}</p>
            </body>
        </html>
        """

        send_html_email(
            RECIPIENT, 
            "AUTOMATIC JOBS - FAILED strategy_trades_daily_update", "", html_content, "",
                email_password=EMAIL_PASSWORD

        )

2024-05-23T00:00:00.000
481
Connected to the database postgre

                MERGE INTO "algo"."strategy_trades" AS tgt
                USING "algo"."stage_strategy_trades" AS src
                ON (tgt.trade_id = src.trade_id)
                WHEN MATCHED THEN UPDATE SET
                "trade_id" = CASE WHEN src."trade_id" IS NOT NULL THEN src."trade_id" ELSE tgt."trade_id" END,
"order_id" = CASE WHEN src."order_id" IS NOT NULL THEN src."order_id" ELSE tgt."order_id" END,
"exchange" = CASE WHEN src."exchange" IS NOT NULL THEN src."exchange" ELSE tgt."exchange" END,
"execution_time" = CASE WHEN src."execution_time" IS NOT NULL THEN src."execution_time" ELSE tgt."execution_time" END,
"state" = CASE WHEN src."state" IS NOT NULL THEN src."state" ELSE tgt."state" END,
"algo_id" = CASE WHEN src."algo_id" IS NOT NULL THEN src."algo_id" ELSE tgt."algo_id" END,
"price" = CASE WHEN src."price" IS NOT NULL THEN src."price" ELSE tgt."price" END,
"quantity" = CASE WHEN src."quantity" IS NOT NU

In [ ]:
# today = datetime(2023,3,18)

# ts_utc = datetime(2023, today.month, today.day, 0, 0, 0).isoformat(timespec='milliseconds')
# print(ts_utc)
# result_dict = cls.get_own_trades(ts_utc)